In [1]:

# =========================================================
# PIPELINE: EDA + PREDICCIÓN + KPIs (R2/MAE/RMSE/MAPE) + EXCEL PARA POWER BI
# Diseñado para correr LOCAL en Windows / OneDrive
# =========================================================

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================================================
# ✅ CONFIG (SOLO EDITA AQUÍ)
# =========================================================
INPUT_FILE = r"C:\Users\EADKD\OneDrive - Bayer\Salesforce\Proyecto Prediccion de datos\report1768425169291.xlsx"

BASE_DIR = os.path.dirname(INPUT_FILE)
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
OUTPUT_FILE = os.path.join(BASE_DIR, "predicciones_powerbi.xlsx")

# Temporadas futuras a predecir: (Temporada_std, Temporada_tipo, Temporada_num)
FUTURE_SEASONS = [
    ("PV 26", "PV", 2026.0),
    ("OI 26/27", "OI", 2026.5),
]

# Parámetro de shrinkage (más alto = más “conservador” hacia ratio global)
K_SHRINK = 3


# =========================================================
# 1) UTILIDADES DE SISTEMA (Windows/OneDrive)
# =========================================================
def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def is_probably_xlsx(file_path: str) -> bool:
    """Un .xlsx real es un ZIP: normalmente empieza con bytes 'PK'."""
    try:
        with open(file_path, "rb") as f:
            head = f.read(4)
        return head.startswith(b"PK")
    except Exception:
        return False


def is_probably_html(file_path: str) -> bool:
    """Detecta si el archivo parece HTML (muchos 'xls' exportados son HTML con <table>)."""
    try:
        with open(file_path, "rb") as f:
            head = f.read(4096).lower()
        return (b"<html" in head) or (b"<table" in head) or head.lstrip().startswith(b"<")
    except Exception:
        return False


def permission_help(action: str, path: str, err: Exception) -> PermissionError:
    return PermissionError(
        f"❌ PermissionError al {action} el archivo:\n"
        f"   {path}\n\n"
        f"✅ Causas típicas:\n"
        f"  1) El archivo está ABIERTO en Excel.\n"
        f"  2) OneDrive lo está sincronizando y lo BLOQUEA.\n"
        f"  3) Power BI tiene el archivo en uso.\n\n"
        f"✅ Soluciones:\n"
        f"  - Cierra Excel y Power BI.\n"
        f"  - Espera a que OneDrive termine de sincronizar.\n"
        f"  - Copia el archivo a C:\\Temp\\ y ejecuta desde ahí.\n\n"
        f"Detalle técnico: {type(err).__name__}: {err}"
    )


# =========================================================
# 2) LECTURA ROBUSTA LOCAL
# =========================================================
def read_html_fallback(file_path: str, original_error: Exception = None) -> pd.DataFrame:
    if original_error:
        print(f"⚠️ Leyendo como HTML (venimos de: {type(original_error).__name__})")

    try:
        df = pd.read_html(file_path, encoding="ISO-8859-1")[0]  # requiere lxml
        df.columns = [c.strip() for c in df.columns]
        print("✅ Lectura HTML OK (lxml)")
        return df
    except ImportError:
        df = pd.read_html(file_path, encoding="ISO-8859-1", flavor="bs4")[0]  # bs4/html5lib
        df.columns = [c.strip() for c in df.columns]
        print("✅ Lectura HTML OK (bs4/html5lib)")
        return df


def load_input(file_path: str) -> pd.DataFrame:
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ No encontré el archivo en: {file_path}")

    ext = os.path.splitext(file_path)[1].lower()
    print(f"📥 Leyendo archivo: {file_path}")

    if ext == ".xlsx":
        if not is_probably_xlsx(file_path) and is_probably_html(file_path):
            return read_html_fallback(file_path)

        try:
            df = pd.read_excel(file_path, engine="openpyxl")
            df.columns = [c.strip() for c in df.columns]
            print("✅ Lectura Excel (.xlsx) OK")
            return df
        except PermissionError as e:
            raise permission_help("leer", file_path, e)
        except Exception as e:
            if is_probably_html(file_path):
                return read_html_fallback(file_path, original_error=e)
            raise RuntimeError(f"❌ Error leyendo .xlsx con openpyxl: {type(e).__name__}: {e}")

    if ext == ".xls":
        if is_probably_html(file_path):
            return read_html_fallback(file_path)

        try:
            df = pd.read_excel(file_path, engine="xlrd")
            df.columns = [c.strip() for c in df.columns]
            print("✅ Lectura Excel (.xls real) OK")
            return df
        except PermissionError as e:
            raise permission_help("leer", file_path, e)
        except Exception as e:
            return read_html_fallback(file_path, original_error=e)

    # otros
    try:
        df = pd.read_excel(file_path)
        df.columns = [c.strip() for c in df.columns]
        print("✅ Lectura Excel genérica OK")
        return df
    except PermissionError as e:
        raise permission_help("leer", file_path, e)


# =========================================================
# 3) LIMPIEZA: numéricos + temporada
# =========================================================
def to_num(s: pd.Series) -> pd.Series:
    s = s.astype(str)
    s = s.str.replace(r"\s+", "", regex=True).str.replace(",", "", regex=False)
    s = s.str.replace(r"[^0-9\.-]+", "", regex=True)
    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    return pd.to_numeric(s, errors="coerce")


def parse_season(label) -> tuple:
    if pd.isna(label):
        return (np.nan, np.nan, np.nan)

    t = str(label).strip().upper()
    t = t.replace("\\", "").replace("–", "-").replace("—", "-")
    t = re.sub(r"\s+", " ", t)

    m = re.match(r"PV\s*(\d{2,4})$", t)
    if m:
        yy = int(m.group(1))
        year = 2000 + yy if yy < 100 else yy
        return (f"PV {str(year)[-2:]}", "PV", float(year))

    m = re.match(r"OI\s*-?\s*(\d{2,4})\s*/\s*(\d{2,4})$", t)
    if m:
        y1 = int(m.group(1))
        y2 = int(m.group(2))
        y1 = 2000 + y1 if y1 < 100 else y1
        y2 = 2000 + y2 if y2 < 100 else y2
        return (f"OI {str(y1)[-2:]}/{str(y2)[-2:]}", "OI", float(y1 + 0.5))

    return (str(label).strip(), np.nan, np.nan)


def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    required = ["Temporada: Temporada", "Nombre del Cliente", "Has Sembradas", "# de bolsas"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"❌ Faltan columnas requeridas: {missing}\nRevisa los nombres exactos del Excel.")

    df = df.copy()
    df["Has Sembradas"] = to_num(df["Has Sembradas"])
    df["# de bolsas"] = to_num(df["# de bolsas"])
    df["Nombre del Cliente"] = df["Nombre del Cliente"].astype(str).str.strip().replace({"nan": np.nan})

    parsed = df["Temporada: Temporada"].apply(parse_season)
    df[["Temporada_std", "Temporada_tipo", "Temporada_num"]] = pd.DataFrame(parsed.tolist(), index=df.index)
    return df


# =========================================================
# 4) EDA
# =========================================================
def eda_report(df: pd.DataFrame, out_dir: str) -> dict:
    ensure_dir(out_dir)

    nulos_pct = (df.isna().mean() * 100).round(2).sort_values(ascending=False).reset_index()
    nulos_pct.columns = ["columna", "%_nulos"]

    numeric_df = df.select_dtypes(include=[np.number])
    stats_num = numeric_df.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.reset_index()
    stats_num = stats_num.rename(columns={"index": "variable"})

    def outlier_iqr(s: pd.Series):
        s = s.dropna().astype(float)
        if len(s) < 8:
            return {"n": len(s), "outliers": 0, "outliers_%": np.nan}
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        out = ((s < low) | (s > high)).sum()
        return {
            "n": len(s), "q1": q1, "q3": q3, "iqr": iqr,
            "low": low, "high": high,
            "outliers": int(out), "outliers_%": round(100 * out / len(s), 2)
        }

    outliers = pd.DataFrame([
        {"variable": "# de bolsas", **outlier_iqr(df["# de bolsas"])},
        {"variable": "Has Sembradas", **outlier_iqr(df["Has Sembradas"])},
    ])

    for col in ["# de bolsas", "Has Sembradas"]:
        plt.figure(figsize=(7, 4))
        x = df[col].dropna()
        plt.hist(x, bins=50, alpha=0.85)
        plt.title(f"Distribución de {col}")
        plt.xlabel(col)
        plt.ylabel("Frecuencia")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"hist_{col.replace('#','num').replace(' ','_')}.png"), dpi=160)
        plt.close()

    return {"nulos_pct": nulos_pct, "stats_num": stats_num, "outliers": outliers}


# =========================================================
# 5) AGREGACIÓN (Power BI friendly)
# =========================================================
def build_agg(df: pd.DataFrame) -> pd.DataFrame:
    df2 = df.dropna(subset=["Nombre del Cliente", "Temporada_num"]).copy()

    def safe_mode(series):
        s = series.dropna()
        return s.mode().iloc[0] if len(s) else np.nan

    agg = (
        df2.groupby(["Nombre del Cliente", "Temporada_std", "Temporada_tipo", "Temporada_num"], as_index=False)
           .agg({
               "# de bolsas": "sum",
               "Has Sembradas": "sum",
               "Regional": safe_mode if "Regional" in df2.columns else "first",
               "Representante": safe_mode if "Representante" in df2.columns else "first",
               "Estado / Departamento": safe_mode if "Estado / Departamento" in df2.columns else "first",
           })
    )
    return agg


# =========================================================
# 6) MÉTRICAS / KPIs (R2 / MAE / RMSE / MAPE)
# =========================================================
def mae(y_true, y_pred) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true, y_pred) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def r2(y_true, y_pred) -> float:
    """
    R2 = 1 - SSE/SST. Puede ser negativo si el modelo es peor que usar la media.
    Devuelve NaN si hay menos de 2 muestras o varianza 0.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    if len(y_true) < 2:
        return np.nan

    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot == 0:
        return np.nan
    return float(1 - ss_res / ss_tot)


def mape(y_true, y_pred) -> float:
    """
    MAPE robusto: ignora y_true=0 para evitar división por cero.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)


def evaluate_ratio_time_cv(agg: pd.DataFrame, k_shrink: int = 3):
    """
    Evaluación tipo 'time-based CV':
    - Para cada temporada s (desde la segunda), entrena con temporadas < s
    - Predice temporada s
    KPIs calculados sobre predicción de # bolsas.
    """
    agg = agg.copy()
    seasons = np.sort(agg["Temporada_num"].dropna().unique())

    fold_rows = []
    pred_rows = []

    # Si hay menos de 2 temporadas, no se puede hacer CV temporal
    if len(seasons) < 2:
        return (
            pd.DataFrame([{"mensaje": "No hay suficientes temporadas para CV temporal"}]),
            pd.DataFrame(),
            pd.DataFrame([{"MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "MAPE_%": np.nan, "n": int(len(agg))}])
        )

    for s in seasons[1:]:
        train = agg[agg["Temporada_num"] < s].copy()
        test = agg[agg["Temporada_num"] == s].copy()

        if train.empty or test.empty:
            continue

        # ratio en train
        train["bags_per_ha"] = train["# de bolsas"] / train["Has Sembradas"]
        train.loc[train["Has Sembradas"] <= 0, "bags_per_ha"] = np.nan

        rtrain = train["bags_per_ha"].dropna()
        if len(rtrain) > 0:
            q1, q3 = rtrain.quantile([0.25, 0.75])
            iqr = q3 - q1
            low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            train["bags_per_ha_clip"] = train["bags_per_ha"].clip(low, high)
        else:
            train["bags_per_ha_clip"] = np.nan

        global_ratio = train.groupby("Temporada_tipo")["bags_per_ha_clip"].median()

        client_ratio = (
            train.groupby(["Nombre del Cliente", "Temporada_tipo"])
                 .agg(median_ratio=("bags_per_ha_clip", "median"), n=("bags_per_ha_clip", "count"))
                 .reset_index()
        )

        # aplicar a test (usa Has real del test)
        test = test.merge(client_ratio, on=["Nombre del Cliente", "Temporada_tipo"], how="left")
        test["global_ratio"] = test["Temporada_tipo"].map(global_ratio)

        ratio_blend = (test["median_ratio"] * test["n"] + test["global_ratio"] * k_shrink) / (test["n"].fillna(0) + k_shrink)
        test["ratio_blend"] = ratio_blend.fillna(test["global_ratio"]).fillna(train["bags_per_ha_clip"].median())

        y_true = test["# de bolsas"].values
        y_pred = np.clip(test["Has Sembradas"].values * test["ratio_blend"].values, 0, None)

        # guardar predicciones
        tmp_preds = test[["Nombre del Cliente", "Temporada_std", "Temporada_tipo", "Temporada_num"]].copy()
        tmp_preds["Bolsas_real"] = y_true
        tmp_preds["Bolsas_pred"] = np.rint(y_pred).astype(int)
        tmp_preds["Fold_test_temporada"] = s
        pred_rows.append(tmp_preds)

        # KPIs fold
        fold_rows.append({
            "temporada_test_num": s,
            "temporada_test_std": test["Temporada_std"].iloc[0] if "Temporada_std" in test.columns and len(test) else str(s),
            "n_test": int(len(y_true)),
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "R2": r2(y_true, y_pred),
            "MAPE_%": mape(y_true, y_pred),
        })

    fold_df = pd.DataFrame(fold_rows)
    preds_df = pd.concat(pred_rows, ignore_index=True) if len(pred_rows) else pd.DataFrame()

    # KPIs globales sobre todos los folds (out-of-sample)
    if not preds_df.empty:
        y_true_all = preds_df["Bolsas_real"].values.astype(float)
        y_pred_all = preds_df["Bolsas_pred"].values.astype(float)
        summary = pd.DataFrame([{
            "MAE": mae(y_true_all, y_pred_all),
            "RMSE": rmse(y_true_all, y_pred_all),
            "R2": r2(y_true_all, y_pred_all),
            "MAPE_%": mape(y_true_all, y_pred_all),
            "n": int(len(y_true_all)),
            "metodo": f"RATIO+SHRINK(k={k_shrink}) - CV Temporal"
        }])
    else:
        summary = pd.DataFrame([{"MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "MAPE_%": np.nan, "n": 0}])

    return fold_df, preds_df, summary


# =========================================================
# 7) PREDICCIÓN FUTURA (Ratio + shrinkage) ✅
# =========================================================
def predict_ratio_method(agg: pd.DataFrame, future_seasons: list, k_shrink: int = 3) -> pd.DataFrame:
    agg = agg.copy()

    agg["bags_per_ha"] = agg["# de bolsas"] / agg["Has Sembradas"]
    agg.loc[agg["Has Sembradas"] <= 0, "bags_per_ha"] = np.nan

    r = agg["bags_per_ha"].dropna()
    if len(r) > 0:
        q1, q3 = r.quantile([0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        agg["bags_per_ha_clip"] = agg["bags_per_ha"].clip(low, high)
    else:
        agg["bags_per_ha_clip"] = np.nan

    global_ratio = agg.groupby("Temporada_tipo")["bags_per_ha_clip"].median()
    global_has = agg.groupby("Temporada_tipo")["Has Sembradas"].median()

    pv_to_oi_factor = global_has.get("OI", np.nan) / global_has.get("PV", np.nan)
    if pd.isna(pv_to_oi_factor) or np.isinf(pv_to_oi_factor):
        pv_to_oi_factor = 1.0

    client_ratio = (
        agg.groupby(["Nombre del Cliente", "Temporada_tipo"])
           .agg(median_ratio=("bags_per_ha_clip", "median"), n=("bags_per_ha_clip", "count"))
           .reset_index()
    )

    latest = agg.sort_values("Temporada_num").groupby("Nombre del Cliente").tail(1)
    latest_info = latest[["Nombre del Cliente", "Regional", "Representante", "Estado / Departamento"]].copy()

    latest_pv = (
        agg[agg["Temporada_tipo"] == "PV"]
        .sort_values("Temporada_num")
        .groupby("Nombre del Cliente").tail(1)[["Nombre del Cliente", "Has Sembradas"]]
        .rename(columns={"Has Sembradas": "Has_PV_ultima"})
    )

    pairs = []
    clients = agg["Nombre del Cliente"].dropna().unique()
    for cl in clients:
        existing = set(agg.loc[agg["Nombre del Cliente"] == cl, "Temporada_std"])
        for (t_std, t_tipo, t_num) in future_seasons:
            if t_std == "PV 26" and "PV 26" in existing:
                continue
            pairs.append((cl, t_std, t_tipo, t_num))

    future = pd.DataFrame(pairs, columns=["Nombre del Cliente", "Temporada_std", "Temporada_tipo", "Temporada_num"])
    future = future.merge(latest_info, on="Nombre del Cliente", how="left")
    future = future.merge(latest_pv, on="Nombre del Cliente", how="left")

    client_has_median = agg.groupby("Nombre del Cliente")["Has Sembradas"].median().rename("Has_mediana")
    future = future.merge(client_has_median, on="Nombre del Cliente", how="left")

    has_pred = np.where(
        future["Temporada_tipo"] == "PV",
        np.where(future["Has_PV_ultima"].notna(), future["Has_PV_ultima"], future["Has_mediana"]),
        np.where(future["Has_PV_ultima"].notna(), future["Has_PV_ultima"] * pv_to_oi_factor, future["Has_mediana"]),
    )
    future["Has_Sembradas_pred"] = np.clip(has_pred, 0, None)

    future = future.merge(client_ratio, on=["Nombre del Cliente", "Temporada_tipo"], how="left")
    future["global_ratio"] = future["Temporada_tipo"].map(global_ratio)

    ratio_blend = (future["median_ratio"] * future["n"] + future["global_ratio"] * k_shrink) / (
        future["n"].fillna(0) + k_shrink
    )
    future["ratio_blend"] = ratio_blend.fillna(future["global_ratio"]).fillna(agg["bags_per_ha_clip"].median())

    bolsas_pred = future["Has_Sembradas_pred"] * future["ratio_blend"]
    future["#_bolsas_pred"] = np.rint(np.clip(bolsas_pred, 0, None)).astype(int)

    return future


# =========================================================
# 8) EXPORT Excel (incluye KPIs)
# =========================================================
def safe_excel_writer(path: str):
    try:
        return pd.ExcelWriter(path, engine="openpyxl")
    except PermissionError as e:
        raise permission_help("escribir", path, e)


def export_excel(df_raw, agg, pred, eda, output_file, kpi_summary, kpi_folds, kpi_preds):
    real_pb = agg.copy().rename(columns={"# de bolsas": "Bolsas", "Has Sembradas": "Has"})
    real_pb["TipoRegistro"] = "Real"

    pred_pb = pred[[
        "Nombre del Cliente", "Regional", "Representante", "Estado / Departamento",
        "Temporada_std", "Temporada_tipo", "Temporada_num"
    ]].copy()
    pred_pb["Has"] = pred["Has_Sembradas_pred"].round(2)
    pred_pb["Bolsas"] = pred["#_bolsas_pred"].astype(int)
    pred_pb["TipoRegistro"] = "Prediccion"

    powerbi = pd.concat([
        real_pb[["Nombre del Cliente","Regional","Representante","Estado / Departamento",
                 "Temporada_std","Temporada_tipo","Temporada_num","Has","Bolsas","TipoRegistro"]],
        pred_pb[["Nombre del Cliente","Regional","Representante","Estado / Departamento",
                 "Temporada_std","Temporada_tipo","Temporada_num","Has","Bolsas","TipoRegistro"]]
    ], ignore_index=True).sort_values(["Nombre del Cliente", "Temporada_num", "TipoRegistro"])

    with safe_excel_writer(output_file) as writer:
        df_raw.to_excel(writer, index=False, sheet_name="Raw")
        agg.to_excel(writer, index=False, sheet_name="Agregado_real")
        pred.to_excel(writer, index=False, sheet_name="Predicciones_detalle")
        powerbi.to_excel(writer, index=False, sheet_name="PowerBI")
        eda["nulos_pct"].to_excel(writer, index=False, sheet_name="EDA_nulos")
        eda["stats_num"].to_excel(writer, index=False, sheet_name="EDA_stats_num")
        eda["outliers"].to_excel(writer, index=False, sheet_name="EDA_outliers")

        # ✅ NUEVO: KPIs del modelo
        kpi_summary.to_excel(writer, index=False, sheet_name="KPI_Modelo")
        kpi_folds.to_excel(writer, index=False, sheet_name="KPI_CV_Detalle")
        kpi_preds.to_excel(writer, index=False, sheet_name="KPI_Pred_vs_Real")


# =========================================================
# 9) MAIN
# =========================================================
def run_pipeline():
    ensure_dir(OUTPUT_DIR)

    # 1) Carga
    df = load_input(INPUT_FILE)

    # 2) Limpieza + temporada
    print("🧹 Preparando datos...")
    df = prepare_dataframe(df)

    # 3) EDA
    print("🔎 Ejecutando EDA...")
    eda = eda_report(df, OUTPUT_DIR)

    # 4) Agregado
    print("🧱 Agregando por Cliente-Temporada...")
    agg = build_agg(df)

    # ✅ 5) KPIs del modelo (calidad)
    print("📏 Calculando KPIs del modelo (CV temporal)...")
    kpi_folds, kpi_preds, kpi_summary = evaluate_ratio_time_cv(agg, k_shrink=K_SHRINK)

    # Muestra rápida en consola (perfecto para video)
    if "R2" in kpi_summary.columns:
        print("\n⭐ KPI Modelo (Global CV):")
        print(kpi_summary.to_string(index=False))

    # 6) Predicción futura
    print("🤖 Generando predicción futura por cliente...")
    pred = predict_ratio_method(agg, FUTURE_SEASONS, k_shrink=K_SHRINK)

    # 7) Export
    print("📤 Exportando Excel para Power BI...")
    export_excel(df, agg, pred, eda, OUTPUT_FILE, kpi_summary, kpi_folds, kpi_preds)

    print("\n✅ LISTO")
    print("📄 Archivo generado:", OUTPUT_FILE)
    print("📁 Gráficas en:", OUTPUT_DIR)
    print("👉 En Power BI importa la hoja: 'PowerBI'")
    print("👉 KPIs del modelo en hojas: KPI_Modelo, KPI_CV_Detalle, KPI_Pred_vs_Real")


if __name__ == "__main__":
    run_pipeline()



📥 Leyendo archivo: C:\Users\EADKD\OneDrive - Bayer\Salesforce\Proyecto Prediccion de datos\report1768425169291.xlsx
✅ Lectura Excel (.xlsx) OK
🧹 Preparando datos...
🔎 Ejecutando EDA...
🧱 Agregando por Cliente-Temporada...
📏 Calculando KPIs del modelo (CV temporal)...

⭐ KPI Modelo (Global CV):
       MAE       RMSE       R2    MAPE_%    n                          metodo
103.062392 189.355259 0.780114 35.373572 1154 RATIO+SHRINK(k=3) - CV Temporal
🤖 Generando predicción futura por cliente...
📤 Exportando Excel para Power BI...

✅ LISTO
📄 Archivo generado: C:\Users\EADKD\OneDrive - Bayer\Salesforce\Proyecto Prediccion de datos\predicciones_powerbi.xlsx
📁 Gráficas en: C:\Users\EADKD\OneDrive - Bayer\Salesforce\Proyecto Prediccion de datos\outputs
👉 En Power BI importa la hoja: 'PowerBI'
👉 KPIs del modelo en hojas: KPI_Modelo, KPI_CV_Detalle, KPI_Pred_vs_Real
